# 02 — Generics: `TypeVar` and `Generic[T]`

When you write a function or class that works *for any type*, you don't want to throw away the type information. Generics let you say "whatever type goes in, the same type comes out."

You've already seen `list[int]` — that's the *use* of a generic class. This notebook is about *writing* them.

## The problem — `Any` throws away types

In [1]:
from typing import Any

def first(xs: list[Any]) -> Any:
    return xs[0]

x = first([1, 2, 3])     # mypy infers Any — we've lost int-ness
y = first(["a", "b"])    # also Any
# x.upper()              # mypy doesn't complain because x is Any — but at runtime: AttributeError

## `TypeVar` — link the input and output type

In [2]:
from typing import TypeVar

# `T` is a placeholder — it gets bound to a concrete type per call.
T = TypeVar("T")

def first(xs: list[T]) -> T:
    return xs[0]

x = first([1, 2, 3])        # mypy: x is int
y = first(["a", "b", "c"])  # mypy: y is str
print(x, type(x).__name__)
print(y, type(y).__name__)
# Now y.upper() works AND mypy will flag x.upper() as an error.

1 int
a str


### The modern syntax (3.12+)

Since Python 3.12, you can declare type parameters inline without `TypeVar`:

In [3]:
# Modern form — clean, no separate TypeVar declaration:
def first_v2[T](xs: list[T]) -> T:
    return xs[0]

print(first_v2([1, 2, 3]))
print(first_v2(["x", "y"]))

1
x


We'll use the 3.12+ syntax in `.py` files. Notebooks and older codebases still show `TypeVar` — recognize both.

## Multiple type variables — `dict[K, V]`-style

In [4]:
def pick[K, V](d: dict[K, V], *keys: K) -> dict[K, V]:
    return {k: d[k] for k in keys if k in d}

u = pick({"name": "alice", "age": 30, "email": "a@b.c"}, "name", "age")
print(u)

scores = {"alice": [90, 85], "bob": [70, 75]}
print(pick(scores, "alice"))
# mypy would reject pick(scores, 42) — 42 is not a str key.

{'name': 'alice', 'age': 30}
{'alice': [90, 85]}


## Bounded type variables — "T must be X or a subclass"

Sometimes you don't want "any" type — you want "any type with a `name` attribute" or "any number".

In [5]:
# Old form using TypeVar with `bound=`:
from typing import TypeVar

NumberT = TypeVar("NumberT", bound=float)

def double(x: NumberT) -> NumberT:
    return x + x

# Modern form (3.12+) — same meaning:
def double_v2[N: float](x: N) -> N:
    return x + x

print(double(5))
print(double(2.5))
# print(double("a"))   # mypy: error — str is not bound by float

10
5.0


## Generic classes — `class Box[T]`

A typed container. After this you can read `list[int]` and `dict[str, V]` definitions for what they really are: parameterized classes.

In [6]:
class Box[T]:
    def __init__(self, value: T) -> None:
        self.value = value

    def get(self) -> T:
        return self.value

    def set(self, value: T) -> None:
        self.value = value

    def __repr__(self) -> str:
        return f"Box({self.value!r})"

b1: Box[int] = Box(42)
b2: Box[str] = Box("hello")
print(b1, b1.get())
print(b2, b2.get().upper())
# b1.set("oops")     # mypy: error — Box[int] cannot accept str

Box(42) 42
Box('hello') HELLO


### A more realistic generic — a typed cache

In [7]:
class Cache[K, V]:
    def __init__(self) -> None:
        self._data: dict[K, V] = {}

    def get(self, key: K) -> V | None:
        return self._data.get(key)

    def set(self, key: K, value: V) -> None:
        self._data[key] = value

    def __len__(self) -> int:
        return len(self._data)

users: Cache[str, dict] = Cache()
users.set("alice", {"id": 1, "role": "admin"})
users.set("bob",   {"id": 2, "role": "user"})
print(len(users))
print(users.get("alice"))
print(users.get("missing"))    # None
# users.set(42, {})     # mypy: error — Cache[str, dict] expects str keys

2
{'id': 1, 'role': 'admin'}
None


## Why this matters for what's next

- **Pydantic / FastAPI** depend heavily on generics — `Response[User]`, `Paginated[Item]`.
- **Repository pattern** (next file in this folder) is `Repository[T]` — one class shape, many entity types.
- **Agent frameworks** use generics for tool schemas, message types, results.

## Mini-exercise

1. Write `last[T](xs: list[T]) -> T | None` returning the last element or `None` if empty.
2. Write `unzip[A, B](pairs: list[tuple[A, B]]) -> tuple[list[A], list[B]]`. Test with `unzip([(1, "a"), (2, "b")])`.
3. Make a generic `Stack[T]` class with `push`, `pop`, `peek`, `__len__`. Annotate carefully — `pop` on an empty stack should return `T | None` or raise; pick one and justify.
4. Why is this typed *wrong*?
   ```python
   def first[T](xs: list[T]) -> T:
       if not xs:
           return None     # ← mypy will flag this. Why?
       return xs[0]
   ```
   Fix it.

In [8]:
# your solutions here


**Takeaways**

- `TypeVar("T")` or `[T]` (3.12+) declares a type parameter.
- A generic function ties input and output types together: `list[T] -> T` is `int` if you pass a `list[int]`.
- Generic classes (`class Cache[K, V]:`) build the typed-container pattern you've been using all along (`list`, `dict`).
- Bounded type vars (`[N: float]`) limit what types can be passed.

Next: [03_protocol_vs_abc.ipynb](03_protocol_vs_abc.ipynb) — the cleanest interface for DI.